In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import time

sys.path.append(os.path.abspath(".."))

RANDOM_STATE = 30

In [3]:
if 'google.colab' in sys.modules: 
    if not os.path.exists('/content/nlp_course_project'):
        !git clone -b lab-08-branch https://github.com/Karoshi-man/nlp_course_project.git
    
    %cd /content/nlp_course_project
    sys.path.append('/content/nlp_course_project')
    
    FOLDER_ID = '1pIDpBFJ33L9XrldgXEXiAnLRNCs6f0gb'
    
    os.makedirs('/content/nlp_course_project/data', exist_ok=True)
    !gdown --folder https://drive.google.com/drive/folders/{FOLDER_ID} -O /content/nlp_course_project/data/
    
    data_dir = '/content/nlp_course_project/data/processed_v2'

else:
    sys.path.append(os.path.abspath('..'))
    data_dir = '../data/processed_v2'

In [4]:
data_file = os.path.join(data_dir, 'processed_v2.csv')
df = pd.read_csv(data_file)

text_col = 'clean_text'

df = df.dropna(subset=[text_col])
print(f"Загальна кількість речень до фільтрації: {len(df)}")

Загальна кількість речень до фільтрації: 583


In [ ]:
from src.topic_modeling import build_lsa_pipeline, build_lda_pipeline
from src.topic_utils import print_top_words, print_top_documents

def count_words(text):
    return len(str(text).split())

In [7]:
df['word_count'] = df[text_col].apply(count_words)
df_filtered = df[df['word_count'] >= 4].copy()
texts = df_filtered[text_col].tolist()

print(f"Документів ДО фільтрації: {len(df)}")
print(f"Документів ПІСЛЯ фільтрації (>=4 слів): {len(texts)}")
print(f"Відкинуто короткого сміття: {len(df) - len(texts)}\n")

MIN_DF = 5
MAX_DF = 0.90
STOP_WORDS = 'english'
NUM_TOPICS_1 = 5
NUM_TOPICS_2 = 8

print("Базові параметри для Векторизаторів (TF-IDF / Count):")
print(f"min_df = {MIN_DF} (відкидаємо рідкісні слова, що є менш ніж у {MIN_DF} документах)")
print(f"max_df = {MAX_DF} (відкидаємо всюдисущі слова, що є у понад {MAX_DF*100}% текстів)")
print(f"stop_words = '{STOP_WORDS}'")

Документів ДО фільтрації: 583
Документів ПІСЛЯ фільтрації (>=4 слів): 583
Відкинуто короткого сміття: 0

Базові параметри для Векторизаторів (TF-IDF / Count):
min_df = 5 (відкидаємо рідкісні слова, що є менш ніж у 5 документах)
max_df = 0.9 (відкидаємо всюдисущі слова, що є у понад 90.0% текстів)
stop_words = 'english'


In [ ]:
# Секція 4: Побудова моделей LSA

start_time = time.time()

# 1. LSA для k=5
lsa_5 = build_lsa_pipeline(n_components=5, min_df=MIN_DF, max_df=MAX_DF, random_state=RANDOM_STATE)
doc_topic_lsa_5 = lsa_5.fit_transform(texts)
print(f"LSA (k=5) успішно навчено!")

# 2. LSA для k=8
lsa_8 = build_lsa_pipeline(n_components=8, min_df=MIN_DF, max_df=MAX_DF, random_state=RANDOM_STATE)
doc_topic_lsa_8 = lsa_8.fit_transform(texts)
print(f"LSA (k=8) успішно навчено!")

print(f"Час навчання LSA: {time.time() - start_time:.2f} секунд")

LSA (k=5) успішно навчено!
LSA (k=8) успішно навчено!
Час навчання LSA: 0.44 секунд


In [13]:
# Секція 5: Побудова моделей LDA

start_time = time.time()

# 1. LDA для k=5
lda_5 = build_lda_pipeline(n_components=5, min_df=MIN_DF, max_df=MAX_DF, random_state=RANDOM_STATE)
doc_topic_lda_5 = lda_5.fit_transform(texts)
print(f"LDA (k=5) успішно навчено!")

# 2. LDA для k=8
lda_8 = build_lda_pipeline(n_components=8, min_df=MIN_DF, max_df=MAX_DF, random_state=RANDOM_STATE)
doc_topic_lda_8 = lda_8.fit_transform(texts)
print(f"LDA (k=8) успішно навчено!")

print(f"Час навчання LDA: {time.time() - start_time:.2f} секунд")

LDA (k=5) успішно навчено!
LDA (k=8) успішно навчено!
Час навчання LDA: 6.36 секунд


In [15]:
# Секція 6: Вивід Топ-слів для LSA та LDA (k=5)

print("ТОП-10 СЛІВ ДЛЯ LSA (k=5)\n" + "-"*40)
lsa_model = lsa_5.named_steps['lsa']
lsa_vec = lsa_5.named_steps['vectorizer']
print_top_words(lsa_model, lsa_vec.get_feature_names_out(), n_top_words=10)

ТОП-10 СЛІВ ДЛЯ LSA (k=5)
----------------------------------------
Тема #0: та data experience ai для team python work на ml
Тема #1: та для на досвід роботи ми даних до або що
Тема #2: data learning machine science analytics etl models pipelines analysis scientist
Тема #3: ai ml learning machine models model vision computer llm science
Тема #4: ai data ти agent workflows product llm build pipelines analytics



In [16]:
print("\nТОП-10 СЛІВ ДЛЯ LDA (k=5)\n" + "-"*40)
lda_model = lda_5.named_steps['lda']
lda_vec = lda_5.named_steps['vectorizer']
print_top_words(lda_model, lda_vec.get_feature_names_out(), n_top_words=10)


ТОП-10 СЛІВ ДЛЯ LDA (k=5)
----------------------------------------
Тема #0: data learning experience ml machine models science computer strong vision
Тема #1: ai experience work team product systems development engineering solutions design
Тема #2: data experience learning teams team security engineering people technical design
Тема #3: та для на досвід ми роботи ai даних до що
Тема #4: experience data python team development work software strong services working



In [17]:
# Секція 7: Вивід топ-документів для тем (Ручна перевірка)

print("ТОП-ДОКУМЕНТИ ДЛЯ LSA (k=5)\n" + "="*50)
print_top_documents(doc_topic_lsa_5, texts, n_top_docs=2)

ТОП-ДОКУМЕНТИ ДЛЯ LSA (k=5)
Найкращі документи для Теми #0
[1] (Вага: 0.517): Sigma Software Всі вакансії компанії
Sigma Software входить до топ 100 найкращих IT компаній світу за рейтингом The Global Outsourcing 100. Компанія входить до складу шведської корпорації Sigma Group, яка налічує 3200 осіб по всьому світу. Понад 1000 проектів реалізовано для клієнтів із Західної Європи, США та України.
Всі вакансії / Data Engineer / Київ
22 січня 2026
Senior Data Engineer (Enterprise and Game Solution Unit)
Київ, Харків, Львів, Дніпро, Одеса, Івано-Франківськ, Луцьк, Будапешт (Угорщина), Бургас (Болгарія), Варшава (Польща), Краків (Польща), Познань (Польща), Прага (Чехія)
Are you an experienced Data Engineer ready to tackle complex, high-load, and data-intensive systems? We are looking for a Senior professional to join our team in Ukraine, Europe, working full-time on a project that will make a real impact in the public sector.
At Sigma Software, we specialize in delivering innovative solutio

In [18]:
print("\nТОП-ДОКУМЕНТИ ДЛЯ LDA (k=5)\n" + "="*50)
print_top_documents(doc_topic_lda_5, texts, n_top_docs=2)


ТОП-ДОКУМЕНТИ ДЛЯ LDA (k=5)
Найкращі документи для Теми #0
[1] (Вага: 0.998): Data Science UA Всі вакансії компанії
Data Science UA is a service company with strong data science and AI expertise. We have been developing AI solutions for all kinds and sizes of businesses. Our expertise includes AI software development, computer vision (image recognition), NLP, machine learning, big data, data analytics, data mining, and data visualization.
Всі вакансії / Data Science / віддалено
28 січня 2026
Senior Data Scientist
віддалено
Data Science UA is a service company with strong data science and AI expertise. Our journey began in 2016 with uniting top AI talents and organizing the first Data Science tech conference in Kyiv. Over the past 9 years, we have diligently fostered one of the largest Data Science & AI communities in Europe.
About the client:
The company is a trailblazer in the world of data-driven advertising, known for its innovative approach to optimizing ad placements and campaign

# Етап 3: Ручна інтерпретація та Аналіз Тем

## 1. Інтерпретація "Хороших" тем
Аналізуючи ключові слова та топ-документи, ми можемо чітко назвати кілька сильних кластерів змісту в ІТ-вакансіях:

* **LSA Тема #2 ("Data Engineering & Analytics"):** * *Слова:* `data, learning, machine, science, analytics, etl, models, pipelines, analysis, scientist`. 
  * *Пояснення:* Ця тема ідеально згрупувала вимоги до Data інженерів та аналітиків. Слова ETL, pipelines та analysis чітко вказують на роботу з інфраструктурою даних.
* **LSA Тема #3 ("Deep Learning & AI Models"):** * *Слова:* `ai, ml, learning, machine, models, model, vision, computer, llm`.
  * *Пояснення:* Вузькоспеціалізована тема для ML-інженерів. На відміну від попередньої, тут фокус на комп'ютерному зорі (computer vision) та великих мовних моделях (LLM).
* **LDA Тема #4 ("Python Backend Development"):** * *Слова:* `experience, data, python, team, development, software, services, working`.
  * *Пояснення:* Класичний опис вакансії Python-розробника. Акцент іде на розробку софту (software), мікросервісів (services) та командну роботу (team), а не на науку про дані.

---

## 2. Аналіз "Поганих / Сміттєвих" тем
Згідно з вимогами, ми ідентифікували явно "шумні" кластери:

* **Погана тема:** LSA Тема #1 та LDA Тема #3.
  * *Слова:* `та, для, на, досвід, роботи, ми, даних, до, або, що`.
  * *Діагностика проблеми:* Це класична "сміттєва тема", яка утворилася виключно з українських прийменників, сполучників та загальних слів. 
  * *Чому так сталося:* Під час препроцесингу ми використали стандартний параметр `stop_words='english'`. Алгоритм успішно видалив англійський шум, але український залишився. Оскільки такі слова часто зустрічаються разом в україномовних описах вакансій (напр., "досвід роботи", "ми шукаємо", "для роботи на..."), модель виділила їх в окрему математичну закономірність.
  * *Як це фіксити далі:* Додати кастомний словник українських стоп-слів у `TfidfVectorizer` та `CountVectorizer`, або проводити попередню лематизацію україномовного тексту.

---

## 3. Битва: LSA vs LDA
* **LSA (TruncatedSVD на TF-IDF)** показала дуже жорсткий поділ за словником. Вона буквально розділила тексти за мовами (окрема тема для українських слів) і дуже чітко розмежувала Data Engineering (ETL) від ML (Computer Vision). Але її теми часто виглядають як просто набори специфічних термінів.
* **LDA (на CountVectorizer)** згенерувала більш "рольові" теми. Наприклад, LDA Тема #1 описує загальний Product/Systems Engineering, а LDA Тема #2 — це явно опис обов'язків Team Lead / Manager (слова: *teams, people, security, design*). 
* **Висновок:** Для аналізу саме *навичок* у коротких реченнях LSA (TF-IDF) здається більш "гострою", оскільки tf-idf "штрафує" часті слова. Проте LDA генерує теми, які більше схожі на реальні описи цілісних вакансій (документ як суміш тем).

## Спеціальний блок: Аналіз "Поганих тем" (Вимога методички)

Окрім сильних та осмислених кластерів, тематичні моделі часто знаходять математичні закономірності, які не мають бізнес-цінності для нашої задачі. Ми виділили мінімум 2 проблемні теми та проаналізували їх природу:

### Погана Тема 1: "Stop-word / Template topic" (Українські прийменники)
* **Де знайдено:** LSA (Тема #1) та LDA (Тема #3)
* **Топ-слова:** `та, для, на, досвід, роботи, ми, даних, до, або, що`.
* **Діагноз:** Це класична тема зі стоп-слів та шаблонних фраз. 
* **Пояснення:** Оскільки вакансії DOU часто пишуться сумішшю англійської (навички) та української (опис компанії), наш параметр `stop_words='english'` прибрав англійський "шум", але залишив український. Модель побачила, що ці слова постійно зустрічаються разом в одному контексті (напр., *"ми пропонуємо для роботи"*, *"досвід роботи від"*), і виділила їх в окрему велику тему. По суті, модель вловила **стиль/мову тексту, а не його зміст**.
* **Рішення:** Додати масив українських стоп-слів у параметри `TfidfVectorizer` та `CountVectorizer`.

### Погана Тема 2: "Mixed / Too generic topic" (Шумний мікс)
* **Де знайдено:** LSA (Тема #0)
* **Топ-слова:** `та, data, experience, ai, для, team, python, work, на, ml`.
* **Діагноз:** Змішана та надто загальна тема.
* **Пояснення:** Ця тема намагається поєднати непоєднуване. Ми бачимо тут українські прийменники (`та, для, на`) разом із загальними англійськими словами ІТ-сфери (`experience, team, work`) та конкретними навичками (`python, ai, ml, data`). Ця тема не дає ніякого розуміння специфіки вакансії. Вона виникла через те, що $k=5$ може бути замалим для LSA: алгоритму не вистачило "кошиків", і він скинув найчастотніші слова, які залишилися після формування вузьких тем, в один загальний "смітник".
* **Рішення:** Збільшити кількість тем (наприклад, подивитися на $k=8$ або $k=10$), а також застосувати більш жорсткий фільтр `max_df` (наприклад, 0.7 замість 0.9), щоб відсіяти загальні слова типу *experience* та *team*.

### Діагностика причин "поганих" тем (п. 6.2)
Аналізуючи наші слабкі кластери (Тема #1 LSA та Тема #0 LSA), ми можемо чітко розкласти причини їх виникнення за ключовими факторами:

* **Проблема в стоп-словах чи шаблонній лексиці?** Абсолютно так. Головна проблема Теми #1 (LSA) — це українські стоп-слова ("та", "для", "на"). Оскільки ми використовували лише `stop_words='english'`, модель виділила українську мову як окрему "тему". Також дуже впливає шаблонна лексика вакансій (фрази типу "досвід роботи від", "ми шукаємо").
* **Проблема в k (кількості тем)?** Так, змішана Тема #0 (де змішалися `data`, `ai`, `team`, `для`) виникла через замале $k=5$. Алгоритму просто не вистачило "розміру", щоб рознести ці концепти по різних кошиках, тому він зліпив їх у загальний кластер.
* **Проблема в preprocessing (фільтрації)?** Частково. Загальні слова типу `experience`, `work`, `team` утворили шум, бо наш параметр `max_df=0.90` виявився надто м'яким. У контексті вакансій ці слова зустрічаються майже скрізь. Їх варто було б відсіяти жорсткішим порогом (напр., `max_df=0.60`).
* **Проблема в неоднорідності корпусу?** Так. Корпус DOU містить суміш суто технічних англомовних термінів (Hard skills) та україномовних описів компанії (Soft skills / Бенефіти). Ця двомовність змушує модель витрачати ресурс на поділ за мовою, а не за суттю навичок.
* **Проблема в надто коротких документах?** Це фундаментальна проблема нашого датасету. Оскільки ми розбили вакансії на окремі речення, документам часто бракує контексту (co-occurrence). LSA і LDA найкраще працюють, коли слова зустрічаються разом у великих абзацах. На коротких реченнях (навіть відфільтрованих `>4` слів) алгоритму складно побудувати стійкі зв'язки між термінами.

### Наступні кроки: Що б ми змінили далі? (п. 6.3)
Для того, щоб покращити якість тематичного моделювання і позбутися виявлених проблем, ми пропонуємо наступні кроки:

**Для виправлення Теми #1 (Stop-word / Template topic):**
1. **Покращити stop-word filtering:** Це найочевидніший крок. Потрібно створити кастомний список українських стоп-слів (напр., `['та', 'для', 'на', 'ми', 'що', 'до']`) та об'єднати його з англійським перед передачею у `vectorizer`.
2. **Перейти на `lemma_text`:** Оскільки українська мова має складну морфологію, лематизація тексту перед векторизацією допоможе звести форми слів (напр., "даних", "дані", "даним") до однієї бази, що зробить теми чистішими.

**Для виправлення Теми #0 (Mixed / Too generic topic):**
1. **Змінити $k$ та $max\_df$:** Необхідно збільшити кількість тем (наприклад, $k=10$ або $k=15$), щоб дати моделі можливість рознести "data" і "team" у різні кластери. Одночасно треба знизити $max\_df$ (наприклад, з 0.90 до 0.60 або 0.70), щоб жорстко відсіяти всюдисущі слова на кшталт "experience" та "work", які "склеюють" різні сюжети разом.
2. **Додати bigrams:** Зміна `ngram_range` на `(1,2)` або `(2,2)` могла б допомогти моделі ловити стійкі словосполучення (напр., "machine learning", "team player", "досвід роботи"), замість того, щоб змішувати їхні окремі слова у загальний кошик.

## Секція 8: Про метрику Coherence (Узгодженість)

У тематичному моделюванні часто використовують метрику **Topic Coherence** (наприклад, c_v або u_mass), щоб автоматично підібрати оптимальну кількість тем ($k$). Вона оцінює, наскільки часто топові слова теми зустрічаються разом у корпусі.

Проте для цієї лабораторної роботи ми свідомо зробили акцент на **ручній оцінці та аналізі топ-документів**. 

**Чому ручна інтерпретація важливіша?**
1. Coherence є лише математичним сигналом. Тема зі стоп-слів (напр., наша Тема #1 з прийменниками "та", "для", "на") може мати дуже високий coherence score просто тому, що ці слова постійно стоять поруч в українських реченнях. Але бізнес-цінність такої теми дорівнює нулю.
2. Для коротких текстів (як наші речення з вакансій DOU) coherence часто працює нестабільно через малий розмір "вікна" контексту (слів у реченні мало, тому co-occurrence матриця дуже розріджена).

**Висновок:** Coherence — це хороший додатковий інструмент для автоматичного відсіювання зовсім "поганих" $k$, але він ніколи не замінить ручного читання реальних документів (top documents) для підтвердження того, що тема дійсно осмислена і не є "сміттєвою".

In [20]:
# Секція 9: Generate docs/audit_summary_lab8.md

os.makedirs("../docs", exist_ok=True)

audit_text = """# Audit Summary: Lab 8 (Topic Modeling)

1. **Завдання:** Знайти приховані сюжети/ролі в реченнях з ІТ-вакансій DOU (Unsupervised learning).
2. **Фільтрація:** Відкинуто короткі речення (< 4 слів). Параметри векторизаторів: `min_df=5`, `max_df=0.90`, `stop_words='english'`.
3. **Моделі:** LSA (TF-IDF + TruncatedSVD) та LDA (CountVectorizer + LatentDirichletAllocation). Перевірено для $k=5$ та $k=8$.
4. **Успішні теми:** Моделі знайшли чіткі кластери для Data Engineering/Analytics, Machine Learning/AI та Python Backend.
5. **Погані теми (Аналіз):**
   - **Stop-word topic:** Виділилася тема суто з українських прийменників ("та", "для", "на"). Причина: `stop_words='english'` не почистив українську частину.
   - **Mixed topic:** Злиття "data", "team" та загальних ІТ-слів. Причина: Замале $k=5$ та занадто м'який поріг `max_df=0.90`.
6. **LSA vs LDA:** Для корпусу коротких речень з вакансій (мікс української та англійської) **LDA виявилася більш читабельною**. LSA створила більше структурного шуму, жорстко розбивши тексти за мовною ознакою.
7. **Coherence:** Свідомо не використовувалась як основний критерій, оскільки для коротких текстів і шумних датасетів ручний аналіз top documents є значно надійнішим (тема зі стоп-слів може мати високий coherence, але не мати бізнес-змісту).
"""

with open("../docs/audit_summary_lab8.md", "w", encoding="utf-8") as f:
    f.write(audit_text)
print("Файл docs/audit_summary_lab8.md успішно згенеровано!")

Файл docs/audit_summary_lab8.md успішно згенеровано!
